In [ ]:
import os
import json

with open('config.json', 'r') as file:
    config = json.load(file)
    togetherAI_api_key = config["togetherAI_api_key"]
    OPEN_AI_API_KEY = config["OpenAI_api_key"]
    weather_api_key = config["weather"]
    XAI_API_KEY = config["grok"]

os.environ['TOGETHER_API_KEY'] = togetherAI_api_key

In [ ]:
!unzip track_b.zip

Archive:  track_b.zip
   creating: track_b/
  inflating: track_b/.DS_Store       
   creating: track_b/dev/
  inflating: track_b/dev/amh.csv     
  inflating: track_b/dev/arq.csv     
  inflating: track_b/dev/chn.csv     
  inflating: track_b/dev/deu.csv     
  inflating: track_b/dev/eng.csv     
  inflating: track_b/dev/esp.csv     
  inflating: track_b/dev/hau.csv     
  inflating: track_b/dev/ptbr.csv    
  inflating: track_b/dev/rus.csv     
  inflating: track_b/dev/ukr.csv     
   creating: track_b/train/
  inflating: track_b/train/amh.csv   
  inflating: track_b/train/arq.csv   
  inflating: track_b/train/chn.csv   
  inflating: track_b/train/deu.csv   
  inflating: track_b/train/eng.csv   
  inflating: track_b/train/esp.csv   
  inflating: track_b/train/hau.csv   
  inflating: track_b/train/ptbr.csv  
  inflating: track_b/train/rus.csv   
  inflating: track_b/train/ukr.csv   


In [ ]:
import pandas as pd
import os

# Define the directory where the files are located
data_dir = "track_b/train/"
output_file = "combined.csv"

# Initialize an empty DataFrame to store combined samples
combined_data = pd.DataFrame()

# Iterate through each CSV file in the directory
for file in os.listdir(data_dir):
    if file.endswith(".csv"):  # Ensure it's a CSV file
        file_path = os.path.join(data_dir, file)
        # Read the CSV file
        data = pd.read_csv(file_path)

        data["lang"] = file_path
        # Take 5 random samples from the file
        sample = data.sample(n=5, random_state=42)
        # Append the samples to the combined DataFrame
        combined_data = pd.concat([combined_data, sample], ignore_index=True)

# Save the combined samples to a new CSV file
combined_data.to_csv(output_file, index=False)

# Read the combined CSV file into a DataFrame
combined_df = pd.read_csv(output_file)

# Display the combined DataFrame
print(combined_df)


                          id  \
0    deu_train_track_b_00782   
1    deu_train_track_b_00959   
2    deu_train_track_b_01826   
3    deu_train_track_b_00421   
4    deu_train_track_b_01939   
5    esp_train_track_b_00890   
6    esp_train_track_b_01675   
7    esp_train_track_b_00415   
8    esp_train_track_b_01428   
9    esp_train_track_b_00850   
10   chn_train_track_b_01599   
11   chn_train_track_b_02309   
12   chn_train_track_b_02387   
13   chn_train_track_b_00979   
14   chn_train_track_b_00369   
15   arq_train_track_b_00071   
16   arq_train_track_b_00236   
17   arq_train_track_b_00431   
18   arq_train_track_b_00413   
19   arq_train_track_b_00040   
20   amh_train_track_b_00300   
21   amh_train_track_b_02551   
22   amh_train_track_b_03082   
23   amh_train_track_b_02343   
24   amh_train_track_b_01212   
25   hau_train_track_b_01345   
26   hau_train_track_b_01935   
27   hau_train_track_b_00129   
28   hau_train_track_b_00333   
29   hau_train_track_b_01164   
30  ptbr

In [ ]:
from openai import OpenAI


# together.ai API Using OpenAI client
# client_together = OpenAI(
#   api_key=os.environ["TOGETHER_API_KEY"],
#   base_url="https://api.together.xyz/v1"
# )

# together.ai API Using together.ai client
# from together import Together
# client_together = Together()

# OpenAI API Using OpenAI client
from openai import OpenAI
client_openai = OpenAI(
  api_key=OPEN_AI_API_KEY
)

MODEL_NAME = "gpt-4o"
# grok
# client_grok = OpenAI(
#   api_key=XAI_API_KEY,
#   base_url="https://api.x.ai/v1",
# )


In [ ]:
from pydantic import BaseModel, Field

'''
class EmotionRewrittenText(BaseModel):
    original_text: str = Field(description="The original sentence provided by the user.")
    joy: str = Field(description="The sentence rewritten to convey joy.")
    sadness: str = Field(description="The sentence rewritten to convey sadness.")
    fear: str = Field(description="The sentence rewritten to convey fear.")
    anger: str = Field(description="The sentence rewritten to convey anger.")
    surprise: str = Field(description="The sentence rewritten to convey surprise.")
    disgust: str = Field(description="The sentence rewritten to convey disgust.")
'''
class EmotionRewrittenText(BaseModel):
    original_text: str
    joy: str
    sadness: str
    fear: str
    anger: str
    surprise: str
    disgust: str


system_prompt_1 = """You are a helpful language model that specializes in changing tone of the given text with different emotional tones. The user will provide an original sentence.
    - joy
    - sadness
    - fear
    - anger
    - surprise
    - disgust

The key requirements are:
1. Maintain the core meaning of the original sentence as closely as possible.
2. Make only minimal and slight changes to convey each specific emotion tone.
3. Keep the sentence length, structure, and vocabulary as similar as you can, introducing only slight modifications to embed each emotional tone.
4. YOU SHOULD NOT EXTEND OR CUT THE ORIGINAL TEXT, YOU ARE ONLY ALLOWED TO HAVE SLIGHT CHANGHES.
"""

def rewrite_into_emotions(question):
    messages=[
            {
                "role": "system",
                "content": system_prompt_1,
            },
            {
                "role": "user",
                "content": question,
            },
        ]
    extract = client_openai.beta.chat.completions.parse(
        messages=messages,
        model=MODEL_NAME,
        response_format=EmotionRewrittenText,

    )
    # completion = extract.parse()
    messages.append({"role": "assistant", "content": extract.choices[0].message.content})
    # messages.append(extract.choices[0].message.to_dict())
    # print(extract.choices[0].message.model_dump())
    output = json.loads(extract.choices[0].message.content)
    return {"messages": messages, "output": output, "usage": extract.usage}


In [ ]:
from pydantic import BaseModel

class EmotionAnalysisNonNestedOutput(BaseModel):
    # Joy
    joy_reasoning: str
    joy_intensity: int

    # Sadness
    sadness_reasoning: str
    sadness_intensity: int

    # Fear
    fear_reasoning: str
    fear_intensity: int

    # Anger
    anger_reasoning: str
    anger_intensity: int

    # Surprise
    surprise_reasoning: str
    surprise_intensity: int

    # Disgust
    disgust_reasoning: str
    disgust_intensity: int


system_prompt_2 = """You are a helpful language model that is determining the emotional intensity of an original sentence across multiple emotions.
You have also been provided with several rewritten versions (one per emotion). These rewrites are not the actual sentences
to score, but rather helper references that demonstrate how the original sentence might look if it were expressed with each
emotion. Focus on the original sentence itself when assigning emotional intensity. The rewrites simply guide your detection.

Instructions:
1. For each emotion (joy, sadness, fear, anger, surprise, disgust), look at the provided emotional rewrite and compare it
   conceptually to the original sentence, treating the rewrite as a clue to see whether the original sentence expresses
   that emotion.
2. Give a brief reasoning on how similar or different the original sentence is from that emotion’s rewrite in terms of
   emotional content.
3. Based on your analysis, assign an intensity class (0, 1, 2, or 3) to the original sentence regarding each emotion:
   - 0: No emotion
   - 1: Low degree of emotion
   - 2: Moderate degree of emotion
   - 3: High degree of emotion"""



def emotion_score(messages):
    # messages.append({
    #             "role": "assistant",
    #             "content": enriched_text,
    #         })
    messages.append({
                "role": "system",
                "content": system_prompt_2,
            })
    extract = client_openai.beta.chat.completions.parse(
        messages=messages,
        model=MODEL_NAME,
        response_format=EmotionAnalysisNonNestedOutput,

    )
    messages.append({"role": "assistant", "content": extract.choices[0].message.content})
    # print(extract.choices[0].message.model_dump())
    output = json.loads(extract.choices[0].message.content)
    return {"messages": messages, "output": output, "usage": extract.usage}

In [ ]:
from copy import deepcopy

def predict_emotions(row):
    ans_dict = deepcopy(row).to_dict()
    question = row["text"]
    # print(ans_dict)
    step_1 = rewrite_into_emotions(question)

    rewritten_forms = step_1["output"]
    step_1_messages = step_1["messages"]

    ans_dict["step_1_prompt_tokens"] = step_1["usage"].prompt_tokens
    ans_dict["step_1_completion_tokens"] = step_1["usage"].completion_tokens
    ans_dict["rewritten_joy"]      = rewritten_forms["joy"]
    ans_dict["rewritten_sadness"]  = rewritten_forms["sadness"]
    ans_dict["rewritten_fear"]     = rewritten_forms["fear"]
    ans_dict["rewritten_anger"]    = rewritten_forms["anger"]
    ans_dict["rewritten_surprise"] = rewritten_forms["surprise"]
    ans_dict["rewritten_disgust"]  = rewritten_forms["disgust"]

    step_2 = emotion_score(step_1_messages)
    step_2_results = step_2["output"]


    ans_dict["step_2_prompt_tokens"] = step_2["usage"].prompt_tokens
    ans_dict["step_2_completion_tokens"] = step_2["usage"].completion_tokens

    ans_dict["reasoning_joy"]      = step_2_results["joy_reasoning"]
    ans_dict["reasoning_sadness"]  = step_2_results["sadness_reasoning"]
    ans_dict["reasoning_fear"]     = step_2_results["fear_reasoning"]
    ans_dict["reasoning_anger"]    = step_2_results["anger_reasoning"]
    ans_dict["reasoning_surprise"] = step_2_results["surprise_reasoning"]
    ans_dict["reasoning_disgust"]  = step_2_results["disgust_reasoning"]

    ans_dict["pred_joy"]      = step_2_results["joy_intensity"]
    ans_dict["pred_sadness"]  = step_2_results["sadness_intensity"]
    ans_dict["pred_fear"]     = step_2_results["fear_intensity"]
    ans_dict["pred_anger"]    = step_2_results["anger_intensity"]
    ans_dict["pred_surprise"] = step_2_results["surprise_intensity"]
    ans_dict["pred_disgust"]  = step_2_results["disgust_intensity"]


    return ans_dict

In [ ]:
results = predict_emotions(combined_df.iloc[0])

[{'role': 'system', 'content': 'You are a helpful language model that specializes in changing tone of the given text with different emotional tones. The user will provide an original sentence.\n    - joy\n    - sadness\n    - fear\n    - anger\n    - surprise\n    - disgust\n\nThe key requirements are:\n1. Maintain the core meaning of the original sentence as closely as possible.\n2. Make only minimal and slight changes to convey each specific emotion tone.\n3. Keep the sentence length, structure, and vocabulary as similar as you can, introducing only slight modifications to embed each emotional tone.\n4. YOU SHOULD NOT EXTEND OR CUT THE ORIGINAL TEXT, YOU ARE ONLY ALLOWED TO HAVE SLIGHT CHANGHES.\n'}, {'role': 'user', 'content': 'Naja als sogenanntes Spielchen, würde ich es persönlich nicht abtun, wie er sagte'}, {'role': 'assistant', 'content': '{"original_text":"Naja als sogenanntes Spielchen, würde ich es persönlich nicht abtun, wie er sagte","joy":"Naja als sogenanntes Spielchen, 

In [ ]:
print(json.dumps(results, indent=4, ensure_ascii=False))

{
    "id": "deu_train_track_b_00782",
    "text": "Naja als sogenanntes Spielchen, würde ich es persönlich nicht abtun, wie er sagte",
    "anger": 0,
    "disgust": 1.0,
    "fear": 0,
    "joy": 0,
    "sadness": 0,
    "surprise": 0,
    "lang": "track_b/train/deu.csv",
    "step_1_prompt_tokens": 288,
    "step_1_completion_tokens": 212,
    "rewritten_joy": "Naja als sogenanntes Spielchen, würde ich es persönlich nicht abtun, denn das klingt doch spannend, wie er sagte",
    "rewritten_sadness": "Naja als sogenanntes Spielchen, würde ich es persönlich nicht abtun, das klang irgendwie trostlos, wie er sagte",
    "rewritten_fear": "Naja als sogenanntes Spielchen, würde ich es persönlich nicht abtun, das hörte sich beunruhigend an, wie er sagte",
    "rewritten_anger": "Naja als sogenanntes Spielchen, würde ich es persönlich nicht abtun, was mich ziemlich wütend macht, wie er sagte",
    "rewritten_surprise": "Naja als sogenanntes Spielchen, würde ich es persönlich nicht abtun, das

In [ ]:
english_df = pd.read_csv("/content/track_b/dev/arq.csv")
english_df

,id,text,anger,disgust,fear,joy,sadness,surprise
0,arq_dev_track_b_00001,الراوي: عيني كانت تشعّل الضو في اللّيل شحال من...,NaN,NaN,NaN,NaN,NaN,NaN
1,arq_dev_track_b_00002,"عيني كملت: ""ما رايحينش نقعدوا هكذا. عويشة تقدر...",NaN,NaN,NaN,NaN,NaN,NaN
2,arq_dev_track_b_00003,مولات الدار: عينك ضرّتنا بزّاف خلاص.,NaN,NaN,NaN,NaN,NaN,NaN
3,arq_dev_track_b_00004,الراوي: واحد الهوا جا على غيلة وجاب شويّة برود...,NaN,NaN,NaN,NaN,NaN,NaN
4,arq_dev_track_b_00005,واحد فيهم ما حلّ فمّه: كانوا مزيّرين على سنانه...,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
95,arq_dev_track_b_00096,منصوريّة: إذا الحياة ولْف، علابالنا من وقتاش و...,NaN,NaN,NaN,NaN,NaN,NaN
96,arq_dev_track_b_00097,الراوي: واحد من الذراري ما نطق.,NaN,NaN,NaN,NaN,NaN,NaN
97,arq_dev_track_b_00098,الراوي: عمر بدا يشوف وجه يماه يبعد. كي كان مكس...,NaN,NaN,NaN,NaN,NaN,NaN
98,arq_dev_track_b_00099,الراوي: من بعد انطلق فالزّنقة حاكماته الخلعة. ...,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
ans = []
for row in english_df.iterrows():
    print(f"{row[0]+1}/{english_df.shape[0]}")
    ans.append(predict_emotions(row[1]))

1/100
2/100
3/100
4/100
5/100
6/100
7/100
8/100
9/100
10/100
11/100
12/100
13/100
14/100
15/100
16/100
17/100
18/100
19/100
20/100
21/100
22/100
23/100
24/100
25/100
26/100
27/100
28/100
29/100
30/100
31/100
32/100
33/100
34/100
35/100
36/100
37/100
38/100
39/100
40/100
41/100
42/100
43/100
44/100
45/100
46/100
47/100
48/100
49/100
50/100
51/100
52/100
53/100
54/100
55/100
56/100
57/100
58/100
59/100
60/100
61/100
62/100
63/100
64/100
65/100
66/100
67/100
68/100
69/100
70/100
71/100
72/100
73/100
74/100
75/100
76/100
77/100
78/100
79/100
80/100
81/100
82/100
83/100
84/100
85/100
86/100
87/100
88/100
89/100
90/100
91/100
92/100
93/100
94/100
95/100
96/100
97/100
98/100
99/100
100/100


In [ ]:
# with open("4o-raw.json", "w") as f:
#     f.write(json.dumps(ans))

In [ ]:
len(ans)

100

In [ ]:
pd.DataFrame(ans).to_csv("4o-arq-full.csv")

In [ ]:
emolist = ["anger", "disgust", "fear", "joy","sadness","surprise"]

In [ ]:
temp_df = pd.DataFrame(ans)[["id"] + [f"pred_{item}" for item in emolist]].reset_index(drop=True)

In [ ]:
temp_df.set_index("id").to_csv("4o-arq.csv")

In [ ]:
# import pandas as pd
# from sklearn.metrics import precision_score, recall_score, f1_score

# # Load the CSV file
# file_path = '4o-english.csv'
# df = pd.read_csv(file_path)

# # Define emotion sets
# emotions = ['joy', 'sadness', 'fear', 'anger', 'surprise', 'disgust']
# emotions_english = ['joy', 'sadness', 'fear', 'anger', 'surprise']

# # Initialize a dictionary to store results for each language
# results_by_lang = {}

# # Group by the 'lang' column
# for lang, group in df.groupby('lang'):
#     # Determine the emotion set based on language
#     if 'eng.csv' in lang.lower():
#         emotion_set = emotions_english
#     else:
#         emotion_set = emotions

#     # Initialize lists for metrics
#     all_true = []
#     all_pred = []
#     precision_per_emotion = []
#     recall_per_emotion = []
#     f1_per_emotion = []

#     # Calculate metrics for each emotion in the selected emotion set
#     for emotion in emotion_set:
#         try:
#             y_true = group[emotion]  # Ground truth labels
#             y_pred = group[f'pred_{emotion}']  # Predicted labels

#             # Drop rows with NaN values
#             # valid_indices = ~(y_true.isna() | y_pred.isna())
#             # y_true = y_true[valid_indices]
#             # y_pred = y_pred[valid_indices]

#             # Append true and predicted labels for micro averaging
#             all_true.extend(y_true)
#             all_pred.extend(y_pred)

#             # Metrics for individual emotions
#             precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
#             recall = recall_score(y_true, y_pred, average='macro')
#             f1 = f1_score(y_true, y_pred, average='macro')

#             # Append to individual scores
#             precision_per_emotion.append(precision)
#             recall_per_emotion.append(recall)
#             f1_per_emotion.append(f1)

#         except Exception as e:
#             print(f"Error calculating metrics for {emotion} in {lang}: {e}")
#             precision_per_emotion.append(0.0)
#             recall_per_emotion.append(0.0)
#             f1_per_emotion.append(0.0)

#     # Calculate macro averages by taking the mean of individual scores
#     macro_precision = sum(precision_per_emotion) / len(precision_per_emotion)
#     macro_recall = sum(recall_per_emotion) / len(recall_per_emotion)
#     macro_f1 = sum(f1_per_emotion) / len(f1_per_emotion)

#     # Calculate micro averages using all combined labels
#     micro_precision = precision_score(all_true, all_pred, average='micro')
#     micro_recall = recall_score(all_true, all_pred, average='micro')
#     micro_f1 = f1_score(all_true, all_pred, average='micro')

#     # Store results for this language
#     results_by_lang[lang] = {
#         'Precision per emotion': precision_per_emotion,
#         'Recall per emotion': recall_per_emotion,
#         'F1 Score per emotion': f1_per_emotion,
#         'Macro Precision': macro_precision,
#         'Macro Recall': macro_recall,
#         'Macro F1': macro_f1,
#         'Micro Precision': micro_precision,
#         'Micro Recall': micro_recall,
#         'Micro F1': micro_f1,
#     }

# # Display results grouped by language
# for lang, metrics in results_by_lang.items():
#     print(f"Results for language: {lang}")
#     print("  Precision per emotion:", metrics['Precision per emotion'])
#     print("  Recall per emotion:", metrics['Recall per emotion'])
#     print("  F1 Score per emotion:", metrics['F1 Score per emotion'])
#     print()
#     print(f"  Macro Precision: {metrics['Macro Precision']:.2f}")
#     print(f"  Macro Recall: {metrics['Macro Recall']:.2f}")
#     print(f"  Macro F1: {metrics['Macro F1']:.2f}")
#     print()
#     print(f"  Micro Precision: {metrics['Micro Precision']:.2f}")
#     print(f"  Micro Recall: {metrics['Micro Recall']:.2f}")
#     print(f"  Micro F1: {metrics['Micro F1']:.2f}")
#     print("=" * 50)


KeyError: 'lang'

In [ ]:
sum(df["step_1_prompt_tokens"] + df["step_2_prompt_tokens"])

66563

In [ ]:
sum(df["step_1_completion_tokens"] + df["step_2_completion_tokens"])

29740

In [ ]:
# df[df["lang"] == "track_b/train/eng.csv"][emotions_english + [f"pred_{emotion}" for emotion in emotions_english]]

,joy,sadness,fear,anger,surprise,pred_joy,pred_sadness,pred_fear,pred_anger,pred_surprise
35,0,0,1,0,0,0,0,0,0,0
36,0,0,2,0,2,0,0,0,0,0
37,0,0,0,0,0,0,0,0,0,0
38,0,1,1,0,0,1,0,0,0,0
39,0,0,1,0,0,0,0,0,0,0


In [ ]:
# import pandas as pd
# from sklearn.metrics import precision_score, recall_score, f1_score

# # Load the CSV file
# file_path = '4o.csv'
# df = pd.read_csv(file_path)

# # List of emotions
# emotions = ['anger','fear','joy','sadness','surprise']

# # Group by 'lang' column
# results_by_lang = {}

# for lang, group in df.groupby('lang'):
#     # Initialize lists to store individual scores for this language
#     all_true = []
#     all_pred = []
#     precision_per_emotion = []
#     recall_per_emotion = []
#     f1_per_emotion = []

#     # Calculate metrics for each emotion
#     for emotion in emotions:
#         try:
#             y_true = group[emotion]  # Ground truth labels
#             y_pred = group[f'pred_{emotion}']  # Predicted labels

#             # Drop rows with NaN values
#             valid_indices = ~(y_true.isna() | y_pred.isna())
#             y_true = y_true[valid_indices]
#             y_pred = y_pred[valid_indices]

#             # Append true and predicted labels for micro averaging
#             all_true.extend(y_true)
#             all_pred.extend(y_pred)

#             # Metrics for individual emotions
#             precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
#             recall = recall_score(y_true, y_pred, average='macro')
#             f1 = f1_score(y_true, y_pred, average='macro')

#             # Append to individual scores
#             precision_per_emotion.append(precision)
#             recall_per_emotion.append(recall)
#             f1_per_emotion.append(f1)

#         except Exception as e:
#             print(f"Error calculating metrics for {emotion} in {lang}: {e}")
#             precision_per_emotion.append(0.0)
#             recall_per_emotion.append(0.0)
#             f1_per_emotion.append(0.0)

#     # Calculate macro averages by taking the mean of individual scores
#     macro_precision = sum(precision_per_emotion) / len(precision_per_emotion)
#     macro_recall = sum(recall_per_emotion) / len(recall_per_emotion)
#     macro_f1 = sum(f1_per_emotion) / len(f1_per_emotion)

#     # Calculate micro averages using all combined labels
#     micro_precision = precision_score(all_true, all_pred, average='micro')
#     micro_recall = recall_score(all_true, all_pred, average='micro')
#     micro_f1 = f1_score(all_true, all_pred, average='micro')

#     # Store the results for this language
#     results_by_lang[lang] = {
#         'Precision per emotion': precision_per_emotion,
#         'Recall per emotion': recall_per_emotion,
#         'F1 Score per emotion': f1_per_emotion,
#         'Macro Precision': macro_precision,
#         'Macro Recall': macro_recall,
#         'Macro F1': macro_f1,
#         'Micro Precision': micro_precision,
#         'Micro Recall': micro_recall,
#         'Micro F1': micro_f1,
#     }

# # Display results grouped by language
# for lang, metrics in results_by_lang.items():
#     print(f"Results for language: {lang}")
#     print("  Precision per emotion:", metrics['Precision per emotion'])
#     print("  Recall per emotion:", metrics['Recall per emotion'])
#     print("  F1 Score per emotion:", metrics['F1 Score per emotion'])
#     print()
#     print(f"  Macro Precision: {metrics['Macro Precision']:.2f}")
#     print(f"  Macro Recall: {metrics['Macro Recall']:.2f}")
#     print(f"  Macro F1: {metrics['Macro F1']:.2f}")
#     print()
#     print(f"  Micro Precision: {metrics['Micro Precision']:.2f}")
#     print(f"  Micro Recall: {metrics['Micro Recall']:.2f}")
#     print(f"  Micro F1: {metrics['Micro F1']:.2f}")
#     print("=" * 50)


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.p

Results for language: track_b/train/amh.csv
  Precision per emotion: [0.0, 0.5, 0.5, 0.5, 1.0]
  Recall per emotion: [0.0, 0.4, 0.4, 0.3333333333333333, 1.0]
  F1 Score per emotion: [0.0, 0.4444444444444444, 0.4444444444444444, 0.375, 1.0]

  Macro Precision: 0.50
  Macro Recall: 0.43
  Macro F1: 0.45

  Micro Precision: 0.60
  Micro Recall: 0.60
  Micro F1: 0.60
Results for language: track_b/train/arq.csv
  Precision per emotion: [0.5833333333333334, 0.8888888888888888, 0.5, 0.4444444444444444, 0.5]
  Recall per emotion: [0.6666666666666666, 0.8333333333333334, 0.3, 0.5555555555555555, 0.6666666666666666]
  F1 Score per emotion: [0.6190476190476191, 0.8222222222222223, 0.375, 0.43333333333333335, 0.5555555555555555]

  Macro Precision: 0.58
  Macro Recall: 0.60
  Macro F1: 0.56

  Micro Precision: 0.68
  Micro Recall: 0.68
  Micro F1: 0.68
Results for language: track_b/train/chn.csv
  Precision per emotion: [0.25, 0.5, 0.5, 0.08333333333333333, 0.5]
  Recall per emotion: [0.125, 0.3, 

In [ ]:
# sum(df["joy"] == df["pred_joy"])

35

In [ ]:
# import pandas as pd
# from sklearn.metrics import precision_score, recall_score, f1_score
# from scipy.stats import pearsonr

# # Load the CSV file
# file_path = '4o.csv'
# df = pd.read_csv(file_path)

# # Define emotion sets
# emotions = ['joy', 'sadness', 'fear', 'anger', 'surprise', 'disgust']
# emotions_english = ['joy', 'sadness', 'fear', 'anger', 'surprise']

# # Initialize a dictionary to store results for each language
# results_by_lang = {}

# # Group by the 'lang' column
# for lang, group in df.groupby('lang'):
#     # Determine the emotion set based on language
#     if 'english' in lang.lower():
#         emotion_set = emotions_english
#     else:
#         emotion_set = emotions

#     # Initialize lists for metrics
#     all_true = []
#     all_pred = []
#     precision_per_emotion = []
#     recall_per_emotion = []
#     f1_per_emotion = []
#     pearson_r_per_emotion = []

#     # Calculate metrics for each emotion in the selected emotion set
#     for emotion in emotion_set:
#         try:
#             y_true = group[emotion]  # Ground truth labels
#             y_pred = group[f'pred_{emotion}']  # Predicted labels

#             # Drop rows with NaN values
#             valid_indices = ~(y_true.isna() | y_pred.isna())
#             y_true = y_true[valid_indices]
#             y_pred = y_pred[valid_indices]

#             # Append true and predicted labels for micro averaging
#             all_true.extend(y_true)
#             all_pred.extend(y_pred)

#             # Metrics for individual emotions
#             precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
#             recall = recall_score(y_true, y_pred, average='macro')
#             f1 = f1_score(y_true, y_pred, average='macro')
#             pearson_r = pearsonr(y_true, y_pred)[0]  # Pearson correlation coefficient

#             # Append to individual scores
#             precision_per_emotion.append(precision)
#             recall_per_emotion.append(recall)
#             f1_per_emotion.append(f1)
#             pearson_r_per_emotion.append(pearson_r)

#         except Exception as e:
#             print(f"Error calculating metrics for {emotion} in {lang}: {e}")
#             precision_per_emotion.append(0.0)
#             recall_per_emotion.append(0.0)
#             f1_per_emotion.append(0.0)
#             pearson_r_per_emotion.append(0.0)

#     # Calculate macro averages by taking the mean of individual scores
#     macro_precision = sum(precision_per_emotion) / len(precision_per_emotion)
#     macro_recall = sum(recall_per_emotion) / len(recall_per_emotion)
#     macro_f1 = sum(f1_per_emotion) / len(f1_per_emotion)
#     avg_pearson_r = sum(pearson_r_per_emotion) / len(pearson_r_per_emotion)

#     # Calculate micro averages using all combined labels
#     micro_precision = precision_score(all_true, all_pred, average='micro')
#     micro_recall = recall_score(all_true, all_pred, average='micro')
#     micro_f1 = f1_score(all_true, all_pred, average='micro')

#     # Store results for this language
#     results_by_lang[lang] = {
#         'Precision per emotion': precision_per_emotion,
#         'Recall per emotion': recall_per_emotion,
#         'F1 Score per emotion': f1_per_emotion,
#         'Pearson r per emotion': pearson_r_per_emotion,
#         'Macro Precision': macro_precision,
#         'Macro Recall': macro_recall,
#         'Macro F1': macro_f1,
#         'Average Pearson r': avg_pearson_r,
#         'Micro Precision': micro_precision,
#         'Micro Recall': micro_recall,
#         'Micro F1': micro_f1,
#     }

# # Calculate total metrics from results_by_lang
# total_precision = []
# total_recall = []
# total_f1 = []
# total_pearson_r = []

# for lang, metrics in results_by_lang.items():
#     total_precision.extend(metrics['Precision per emotion'])
#     total_recall.extend(metrics['Recall per emotion'])
#     total_f1.extend(metrics['F1 Score per emotion'])
#     total_pearson_r.extend(metrics['Pearson r per emotion'])

# # Calculate overall macro scores
# total_macro_precision = sum(total_precision) / len(total_precision)
# total_macro_recall = sum(total_recall) / len(total_recall)
# total_macro_f1 = sum(total_f1) / len(total_f1)
# total_avg_pearson_r = sum(total_pearson_r) / len(total_pearson_r)

# # Display results grouped by language
# for lang, metrics in results_by_lang.items():
#     print(f"Results for language: {lang}")
#     print("  Precision per emotion:", metrics['Precision per emotion'])
#     print("  Recall per emotion:", metrics['Recall per emotion'])
#     print("  F1 Score per emotion:", metrics['F1 Score per emotion'])
#     print("  Pearson r per emotion:", metrics['Pearson r per emotion'])
#     print()
#     print(f"  Macro Precision: {metrics['Macro Precision']:.2f}")
#     print(f"  Macro Recall: {metrics['Macro Recall']:.2f}")
#     print(f"  Macro F1: {metrics['Macro F1']:.2f}")
#     print(f"  Average Pearson r: {metrics['Average Pearson r']:.2f}")
#     print()
#     print(f"  Micro Precision: {metrics['Micro Precision']:.2f}")
#     print(f"  Micro Recall: {metrics['Micro Recall']:.2f}")
#     print(f"  Micro F1: {metrics['Micro F1']:.2f}")
#     print("=" * 50)

# # Display total results
# print("Total Results:")
# print(f"  Macro Precision: {total_macro_precision:.2f}")
# print(f"  Macro Recall: {total_macro_recall:.2f}")
# print(f"  Macro F1: {total_macro_f1:.2f}")
# print(f"  Average Pearson r: {total_avg_pearson_r:.2f}")


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
<ipython-input-289-6628c5a54fb7>:51: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_r = pearsonr(y_true, y_pred)[0]  # Pearson correlation coefficient
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true sam

Error calculating metrics for disgust in track_b/train/eng.csv: x and y must have length at least 2.


<ipython-input-289-6628c5a54fb7>:51: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_r = pearsonr(y_true, y_pred)[0]  # Pearson correlation coefficient
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
<ipython-input-289-6628c5a54fb7>:51: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_r = pearsonr(y_true, y_pred)[0]  # Pearson correlation coefficient
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

Results for language: track_b/train/amh.csv
  Precision per emotion: [0.5, 0.5, 0.5, 0.0, 1.0, 0.0]
  Recall per emotion: [0.4, 0.3333333333333333, 0.4, 0.0, 1.0, 0.0]
  F1 Score per emotion: [0.4444444444444444, 0.375, 0.4444444444444444, 0.0, 1.0, 0.0]
  Pearson r per emotion: [nan, 0.9081082718950219, nan, -0.140028008402801, 1.0, -0.8017837257372731]

  Macro Precision: 0.42
  Macro Recall: 0.36
  Macro F1: 0.38
  Average Pearson r: nan

  Micro Precision: 0.50
  Micro Recall: 0.50
  Micro F1: 0.50
Results for language: track_b/train/arq.csv
  Precision per emotion: [0.5, 0.4444444444444444, 0.8888888888888888, 0.5833333333333334, 0.5, 0.3333333333333333]
  Recall per emotion: [0.3, 0.5555555555555555, 0.8333333333333334, 0.6666666666666666, 0.6666666666666666, 0.3333333333333333]
  F1 Score per emotion: [0.375, 0.43333333333333335, 0.8222222222222223, 0.6190476190476191, 0.5555555555555555, 0.3333333333333333]
  Pearson r per emotion: [nan, 0.6123724356957946, 0.8451542547285165, 

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
<ipython-input-289-6628c5a54fb7>:51: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_r = pearsonr(y_true, y_pred)[0]  # Pearson correlation coefficient
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true sam